#**Project Overview**

**Goal**: Create an AI assistant that answers questions about a collection of documents (e.g., AI ethics articles). It retrieves relevant context semantically, generates safe responses, extracts structured metadata, and uses an agent for tasks like calculations.

**Tech Stack:**

1. LangChain (for chains, agents, prompts).
2. Cohere LLM (via langchain-cohere).
3. Sentence Transformers & FAISS (for embeddings and vector DB).
4. Pydantic (for structured output validation).

**Dataset**: Use a small set of sample documents (expandable). For example, AI-related texts.

##**Step 1: Environment Setup (A - Foundations)**

Install libraries and set up the LLM.

In [ ]:
!pip install -q langchain langgraph langchain-core langchain-community langchain-cohere sentence-transformers faiss-cpu pydantic cohere torch  # faiss-gpu possible avec un GPU Nvidia


In [ ]:
import os
from langchain_cohere import ChatCohere
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field

In [ ]:
# Clé API Cohere : JAMAIS écrite en clair dans le code.
# - Sur Colab : ajoute-la dans les « Secrets » (icône 🔑 à gauche) sous le nom COHERE_API_KEY
# - Ailleurs : définis la variable d'environnement COHERE_API_KEY, ou saisis-la ci-dessous
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["COHERE_API_KEY"] = userdata.get("COHERE_API_KEY")
except Exception:
    if not os.getenv("COHERE_API_KEY"):
        os.environ["COHERE_API_KEY"] = getpass("Clé API Cohere : ")

# Initialize Cohere LLM
llm = ChatCohere(model="command-r-plus-08-2024", temperature=0.0)

# Initialize embedding model (semantic search)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')


##**Step 2: Data Ingestion & Vector Database (B - Semantic Search)**

Load documents and build a vector DB

In [ ]:
# Sample documents (expand with your course notes or PDFs)
documents = [
    "AI ethics involves principles for responsible AI development.",
    "Large Language Models (LLMs) like GPT can generate human-like text but raise bias concerns.",
    "Nvidia GPUs accelerate deep learning training by parallelizing computations.",
    "Retrieval-Augmented Generation (RAG) combines search with generation to reduce hallucinations.",
    "Guardrails in AI prevent harmful outputs, such as detecting unsafe queries."
]

# Generate embeddings
embeddings = embedding_model.encode(documents)

# Convert to float32 for FAISS (Nvidia-optimized for large datasets)
embeddings_np = np.array(embeddings).astype('float32')

# Create FAISS index (use faiss.IndexFlatL2 or IndexIVFFlat for scale)
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance
index.add(embeddings_np)  # Add vectors

print(f"Vector DB built with {index.ntotal} documents.")

## What We Did Here

### 1. Created Sample Documents
We defined a small list of AI-related text documents.  
These act as our knowledge base (like course notes or PDF content).

---

### 2. Converted Text to Embeddings
We used an embedding model to transform each document into a numerical vector.

Why?
- Machines cannot understand raw text.
- Embeddings convert text into numbers while preserving meaning.
- Similar texts will have similar vectors.

---

### 3. Prepared Data for FAISS
We converted embeddings to `float32` format.

Why?
- FAISS (Facebook AI Similarity Search) requires float32 vectors.
- This format is optimized for fast similarity search (especially with Nvidia GPUs).

---

### 4. Built a FAISS Vector Index
We:
- Determined the vector dimension.
- Created a `faiss.IndexFlatL2` index (uses L2 distance).
- Added all document vectors to the index.

This creates a **Vector Database**.

---

### 5. Result
We now have a searchable vector database.

This allows us to:
- Search for similar documents.
- Retrieve relevant context.
- Use it in systems like RAG (Retrieval-Augmented Generation).

In short:
We converted text → embeddings → stored them in a FAISS vector database for semantic search.


##**Step 3: Build Retrieval Chain with LCEL (C - RAG Basics)**

Create a retrieval function and chain it with generation using LCEL.

In [ ]:
# Retrieval tool (semantic search)
@tool
def retrieve_documents(query: str, k: int = 2) -> str:
    """Retrieve top-k relevant documents from the vector DB."""
    query_embedding = embedding_model.encode([query])
    query_np = np.array(query_embedding).astype('float32')
    D, I = index.search(query_np, k)  # Distances and indices
    return " ".join([documents[i] for i in I[0] if i >= 0])  # Combine docs

# LCEL Chain: Prompt -> LLM -> Parser
rag_prompt = PromptTemplate.from_template(
    """Context: {context}
    Question: {question}
    Answer concisely based on context."""
)

rag_chain = rag_prompt | llm | StrOutputParser()

## What We Did Here

## 1. Created a Retrieval Tool (Semantic Search)

We defined a tool called `retrieve_documents`.

Its role is to retrieve the most relevant documents from the vector database.

### How it works:

1. Convert the user query into an embedding (vector representation).
2. Convert it to `float32` (required by FAISS).
3. Use `index.search()` to find the top-k closest vectors.
4. Retrieve the corresponding original documents.
5. Combine them into a single context string.

This allows **semantic search** (search by meaning, not keywords).

---

## 2. Built the RAG Prompt

We created a prompt template:

Context: {context}  
Question: {question}  
Answer concisely based on context.

This ensures:
- The model receives retrieved context.
- The answer is grounded in that context.
- The response is concise and focused.

---

## 3. Created the LCEL RAG Chain

We built a LangChain Expression Language (LCEL) pipeline:

Prompt → LLM → Output Parser

Specifically:

- `rag_prompt` formats the input.
- `llm` generates the answer.
- `StrOutputParser()` ensures the output is a clean string.

---

# Final Flow

User Query  
→ Retrieve Relevant Documents  
→ Inject Context into Prompt  
→ LLM Generates Answer  
→ Return Clean Output  

This is the core of a basic **Retrieval-Augmented Generation (RAG)** system.


##**Step 4: Add Guardrails for Safety (D - Input Validation)**

Implement input safety checks before any LLM call.

In [ ]:
# Unsafe patterns (expand as needed)
UNSAFE_PATTERNS = [
    r"how to.*(kill|hurt|harm)",
    r"self harm|suicide",
    r"bomb|weapon|illegal"
]

def is_unsafe_input(text: str) -> bool:
    text = text.lower()
    return any(re.search(p, text) for p in UNSAFE_PATTERNS)

def safe_rag_invoke(query: str):
    if is_unsafe_input(query):
        return "Sorry, I can't respond to that. Please ask something else."
    context = retrieve_documents(query)
    return rag_chain.invoke({"context": context, "question": query})

# Unsafe Input Handling (Guardrails)

## 1. Defined Unsafe Patterns

We created a list of regex patterns to detect unsafe or harmful queries:

- Violence-related requests: `kill`, `hurt`, `harm`
- Self-harm or suicide-related terms
- Bombs, weapons, or illegal activities

This acts as a **basic content moderation layer**.

---

## 2. Built an Input Safety Checker

```python
def is_unsafe_input(text: str) -> bool:
    text = text.lower()
    return any(re.search(p, text) for p in UNSAFE_PATTERNS)
```

How it works:

- Converts input to lowercase.
- Checks if any unsafe pattern matches.
- Returns `True` if unsafe, otherwise `False`.

This prevents dangerous queries from reaching the model.

---

## 3. Wrapped RAG with Safety

```python
def safe_rag_invoke(query: str):
    if is_unsafe_input(query):
        return "Sorry, I can't respond to that. Please ask something else."
    context = retrieve_documents(query)
    return rag_chain.invoke({"context": context, "question": query})
```

Workflow:

1. Check if the query is unsafe.
   - If unsafe → return a refusal message.
2. If safe:
   - Retrieve relevant documents.
   - Invoke the RAG chain with context and question.
   - Return the generated answer.

---

## 4. Result

We now have a **safe RAG system**:

User Query  
→ Safety Check  
→ (If safe) Retrieve Context  
→ Generate Answer  
→ Return Response  

This ensures harmful or unsafe queries are blocked before reaching the LLM.


##**Step 5: Extract Structured Output (E - JSON Parsing & Validation)**

Force JSON output and validate with Pydantic.

In [ ]:
# Pydantic model for structured response
class StructuredAnswer(BaseModel):
    answer: str = Field(description="Concise answer")
    sources: list[str] = Field(description="List of source documents")
    confidence: float = Field(description="Confidence score (0-1)")

# JSON Parser
json_parser = JsonOutputParser(pydantic_object=StructuredAnswer)

# Updated prompt for structured output
structured_prompt = ChatPromptTemplate.from_template(
    """Context: {context}
    Question: {question}
    Respond in JSON format: {format_instructions}"""
)

# Chain with JSON enforcement
structured_chain = structured_prompt | llm | json_parser

def structured_rag(query: str):
    if is_unsafe_input(query):
        return {"error": "Unsafe query"}
    context = retrieve_documents(query)
    return structured_chain.invoke({
        "context": context,
        "question": query,
        "format_instructions": json_parser.get_format_instructions()
    })

# Structured RAG with Pydantic Validation

## 1. Defined a Structured Output Schema

We created a `StructuredAnswer` Pydantic model:

- `answer`: Concise answer from the LLM
- `sources`: List of documents used as context
- `confidence`: Confidence score between 0 and 1

This enforces a **structured JSON output** rather than free text.

---

## 2. Added a JSON Output Parser

```python
json_parser = JsonOutputParser(pydantic_object=StructuredAnswer)
```

- Validates the LLM output against the Pydantic schema
- Ensures the response is consistent and structured
- Catches formatting errors automatically

---

## 3. Updated Prompt for Structured Responses

We modified the prompt to instruct the LLM:

```
Context: {context}
Question: {question}
Respond in JSON format: {format_instructions}
```

- `{format_instructions}` ensures the output matches the schema
- Guides the LLM to include `answer`, `sources`, and `confidence`

---

## 4. Built the Structured RAG Chain

```python
structured_chain = structured_prompt | llm | json_parser
```

Pipeline:

1. Format the prompt
2. Generate response via LLM
3. Parse and validate as structured JSON

---

## 5. Combined Safety + Structured Output

```python
def structured_rag(query: str):
    if is_unsafe_input(query):
        return {"error": "Unsafe query"}
    context = retrieve_documents(query)
    return structured_chain.invoke({
        "context": context,
        "question": query,
        "format_instructions": json_parser.get_format_instructions()
    })
```

- Checks for unsafe queries
- Retrieves relevant documents
- Invokes the structured RAG chain
- Returns a validated JSON object

---

## 6. Result

We now have a **safe, structured RAG system**:

User Query  
→ Safety Check  
→ Retrieve Context  
→ Generate Structured JSON  
→ Return Validated Output  

This is production-ready, reliable, and interpretable.


##**Step 6: Build an Agent for Multi-Step Reasoning (F - Agents)**

Add an agent with tools (e.g., calculator + retrieval).

In [ ]:
import ast
import operator
from langgraph.checkpoint.memory import MemorySaver

# Calculatrice sécurisée : on analyse l'expression au lieu d'utiliser eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg}

def _safe_eval(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("only numbers and arithmetic operators are allowed")

@tool
def calculator(expression: str) -> str:
    """Compute an arithmetic expression, e.g. '5814 * 6884'."""
    try:
        expr = expression.replace("×", "*").replace("^", "**")
        return str(_safe_eval(ast.parse(expr, mode="eval").body))
    except Exception as e:
        return f"Error: {str(e)}"

tools = [calculator, retrieve_documents]

# Agent ReAct avec mémoire (checkpointer) : il choisit lui-même quand appeler un outil
checkpointer = MemorySaver()
agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=checkpointer,
    system_prompt=(
        "You are a helpful assistant. For questions about AI, LLMs, RAG, ethics, guardrails "
        "or GPUs, call retrieve_documents and base your answer on the returned passages. "
        "For arithmetic, call calculator. Answer concisely."
    ),
)

def agent_invoke(query: str, thread_id: str = "default"):
    if is_unsafe_input(query):
        return "Unsafe query detected."
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke({"messages": [HumanMessage(content=query)]}, config=config)
    return result["messages"][-1].content


# Stateful ReAct Agent with Tools and Memory

## 1. Added Tools to the Agent

We defined two tools:

- **calculator** → Evaluates math expressions safely by parsing them with `ast` (no `eval`)
- **retrieve_documents** → Performs semantic search in the vector database

These tools allow the agent to:

- Perform reasoning
- Call external functions dynamically
- Combine retrieval and computation in responses

---

## 2. Created a ReAct Agent

```python
agent = create_agent(model=llm, tools=tools)
```

- Follows the **ReAct pattern**: Think → Choose Tool → Call Tool → Observe → Answer
- The agent can decide at runtime which tool to use
- Supports multi-step reasoning and dynamic decision-making

---

## 3. Added Memory with a Checkpointer

```python
checkpointer = MemorySaver()
agent = create_agent(model=llm, tools=tools, checkpointer=checkpointer)
```

- Stores conversation state
- Enables multi-turn interactions
- Supports resuming previous conversations
- Each conversation can be tracked independently

---

## 4. Enabled Thread-Based Memory

```python
def agent_invoke(query: str, thread_id: str = "default"):
    if is_unsafe_input(query):
        return "Unsafe query detected."
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke({"messages": [HumanMessage(content=query)]}, config=config)
    return result["messages"][-1].content
```

- `thread_id` allows isolating memory per user or session
- Multiple threads can run in parallel without sharing context
- Ensures persistent, contextual conversation per thread

---

## 5. Safety Layer

- Queries are first checked against unsafe patterns
- Unsafe queries are blocked with a warning message
- Safe queries proceed to agent execution

---

## 6. Final System Architecture

User Query  
→ Safety Check  
→ ReAct Agent  
→ Tool Usage (Calculator / Retrieval)  
→ Memory Saved per Thread  
→ Final Answer

---

## 7. Outcome

We now have a **complete conversational AI agent** that is:

- Tool-enabled (retrieval + calculator)
- Memory-aware (per-thread persistent context)
- Multi-turn capable
- Guarded with basic safety checks
- Able to reason dynamically and provide grounded responses


##**Step 7: Integrate Everything into a Full System (G - Coherent Pipeline)**

Combine everything into a single pipeline: **guardrail → ReAct agent (retrieval + calculator + memory)**, and test it on representative queries.


In [ ]:
# Full pipeline: Guardrail -> ReAct agent (semantic retrieval + calculator) with memory
def full_pipeline(query: str, thread_id: str = "demo") -> str:
    return agent_invoke(query, thread_id=thread_id)   # agent_invoke applies the guardrail first

test_queries = [
    "Hi, I'm testing the assistant.",                                  # small talk -> no tool
    "What is RAG?",                                                    # retrieval tool
    "Calculate 5814 * 6884",                                           # calculator tool
    "How to create a bomb?",                                           # blocked by the guardrail
    "Summarize in one sentence what you told me about RAG.",           # uses memory
]

for q in test_queries:
    print(f"Q: {q}")
    print(f"A: {full_pipeline(q)}\n")

# Structured output (JSON validated by Pydantic)
print("Structured answer:", structured_rag("What are guardrails in AI?"))

# Optional: interactive session (set to True to chat freely)
INTERACTIVE = False
if INTERACTIVE:
    print("Safe RAG Agent Assistant with Memory (type 'exit' to quit)")
    while True:
        query = input("\nYour question: ")
        if query.lower() in ["exit", "quit", "q"]:
            break
        print("\nResponse:\n" + full_pipeline(query, thread_id="interactive"))


# Full Pipeline: Guardrails → Agent → Tools → Answer

## 1. One entry point

`full_pipeline(query)` sends every question through the same path:

1. **Safety check** (`is_unsafe_input`): unsafe queries are refused before reaching the LLM
2. **ReAct agent** decides what to do:
   - `retrieve_documents` → semantic search in the FAISS vector database (RAG)
   - `calculator` → exact arithmetic
   - or a direct answer for small talk
3. The LLM writes the final answer **from the tool results**
4. The conversation is saved under a `thread_id` (**memory**)

## 2. Automatic tests instead of a blocking loop

The cell runs representative queries (small talk, RAG, calculation, unsafe request, memory), so the whole notebook can be executed with *Run all*. An interactive mode is still available with `INTERACTIVE = True`.

## 3. Structured output

`structured_rag` returns a JSON answer validated by Pydantic (`answer`, `sources`, `confidence`).

## 4. From notebook to application

The deployable version of this pipeline (API with LangServe + Gradio interface) is in `agent.py`, `server.py` and `app.py` at the root of the repository.
